In [11]:
import random
import pandas as pd
import numpy as np

In [12]:
# 方案一
def num_func(frag_number):
    """
    将数量为 frag_number 的片段随机落入100个抽屉；
    统计落入每个抽屉的片段数量；
    返回：片段数量最多的抽屉的片段数量。
    """
    a = np.random.randint(0,100,size=frag_number)
    b = pd.value_counts(a)
    c = max(b)
    return a,b,c

# 方案二
def num_func_two(frag_number):
    """
    将数量为 frag_number 的片段随机落入100个抽屉；
    统计落入每个抽屉的片段数量；
    返回：片段数量最多、次多的抽屉的片段数量。
    """
    a = np.random.randint(0,100,size=frag_number)
    b = pd.value_counts(a,sort=True)
    c = b.values[0:2]
    return c

In [13]:
frag_number_max = 20000
repeat_times = 10000

单线程：

In [21]:
%%time
freq_df = pd.DataFrame()
### 从 0 开始 make
#for i in range(frag_number_max):
### 从 frag_number_max 开始 make
for i in range(frag_number_max,frag_number_max+8000):
    frag_number = i+1
    
    a = map(num_func,[frag_number]*repeat_times)
    b = pd.value_counts(list(a))
    
    num_func_freq = b/repeat_times
    num_func_freq_df = pd.DataFrame({"frag_num_100k":frag_number,"frag_num_1k_max":num_func_freq.index,"freq":num_func_freq.values})
    num_func_freq_df.sort_values(by="frag_num_1k_max",inplace=True,ascending=False)
    
    freq_sum_temp = []
    freq_sum_cutoff_temp = []
    flag = True
    
    for temp in range(len(num_func_freq_df["freq"])):
        freq_sum_temp.append(sum(num_func_freq_df["freq"][0:temp+1]))
        if(freq_sum_temp[-1] <= 0.05):
            freq_sum_cutoff_temp.append(False)
        elif((freq_sum_temp[-1] > 0.05)&flag):
            freq_sum_cutoff_temp.append(True)
            flag = False
        else:
            freq_sum_cutoff_temp.append(False)
        pass
    num_func_freq_df["freq_sum"] = freq_sum_temp
    num_func_freq_df["freq_sum_cutoff"] = freq_sum_cutoff_temp
    
    freq_df = pd.concat([freq_df, num_func_freq_df])
    pass    

Wall time: 9h 38min 39s


In [22]:
freq_df_cutoff = freq_df[freq_df["freq_sum_cutoff"]]
freq_df.to_csv('./Uniform_Distribution/Uniform_Distribution.csv',index=False)
freq_df_cutoff.to_csv('./Uniform_Distribution/Uniform_Distribution_cutoff.csv',index=False)

多线程：